# Задание 6

### 6. Случайная величина имеет распределение Парето.

\begin{equation*} 
p(x) =
\begin{cases} 
\frac{\theta - 1}{x^{\theta}}, & \text{если } x \geq 1 \\
0, & \text{если } x < 1
\end{cases}
,\theta > 1.
\end{equation*}


#### d)  Сгенеруйте выборку объёмом n = 100 для некоторого значения парамeтра $\theta$. Вычислите указанные выше доверительные интегралы для доверительной веротяности 0.95.

In [1]:
import numpy as np
import random as rd

theta = 2

def p(x, theta):
    if x >= 1:
        return (theta - 1) / (x ** theta)
    return 0

def F(x, theta):
    if x >= 1:
        return 1 - x ** (1 - theta)
    return 0

def F_inv(y, theta):
    return (1 - y) ** (1 / (1 - theta))

sample_size = 100
sample = []

for _ in range(sample_size):
    y = rd.random()
    sample.append(F_inv(y, theta))

Доверительный интервал медианы

$$
P(g(\tilde\theta) - \frac{1.96 * 2^{\frac{1}{\tilde\theta-1}} \ln(2)}{\sqrt{n}(\tilde\theta-1)} < median < g(\tilde\theta) + \frac{1.96 * 2^{\frac{1}{\tilde\theta-1}} \ln(2)}{\sqrt{n}(\tilde\theta-1)})=\beta
$$

$$
P(2 ^ {\overline\ln(x)}- \frac{t_2  2 ^ {\overline\ln(x)} \ln(2) \overline\ln(x)}{\sqrt{n}} < median < 2 ^ {\overline\ln(x)} - \frac{t_1 2 ^ {\overline\ln(x)} \ln(2)  \overline\ln(x)}{\sqrt{n}})=\beta
$$

In [2]:
beta = 0.95

log_mean = np.mean([np.log(x) for x in sample])

g_theta = 2 ** (log_mean)

lower_bound = g_theta - 1.96 * g_theta * np.log(2) * log_mean / np.sqrt(sample_size)
upper_bound = g_theta + 1.96 * g_theta * np.log(2) * log_mean / np.sqrt(sample_size)

print(f"Доверительный интервал: ({lower_bound}, {upper_bound}), длина: {upper_bound - lower_bound}")

variation_row = sorted(sample)
print("Медиана: ", (variation_row[49] + variation_row[50])/2)

Доверительный интервал: (1.6506021723497146, 2.118848063849969), длина: 0.46824589150025453
Медиана:  1.7686399278151774


Асимптотический доверительный интервал

$$
P(1 - \frac{t_2 - \sqrt{n}}{\overline{\ln(x)} \sqrt{n}} < \theta < 1 - \frac{t_1 - \sqrt{n}}{\overline{\ln(x)} \sqrt{n}} )=\beta
$$

In [3]:
beta = 0.95

lower_bound = 1 - (1.96 - np.sqrt(sample_size)) / (log_mean * np.sqrt(sample_size))
upper_bound = 1 + (1.96 + np.sqrt(sample_size)) / (log_mean * np.sqrt(sample_size))

print(f"Доверительный интервал: ({lower_bound}, {upper_bound})")
print(f"Длина: {upper_bound - lower_bound}")

Доверительный интервал: (1.8793092056159868, 2.3080271267620898)
Длина: 0.42871792114610296


#### t)  Численно постройте бутстрапоский доверительный интервал двумя способами, используя параметрический бутстрап и непараметрический бутстрап.

Непараметрический бутстрап

In [4]:
n_iterations = 1000
beta = 0.95


h_wave = 1 + 1 / log_mean
bootstrap_delta = []

for _ in range(n_iterations):
    bootstrap_sample = np.random.choice(sample, size=sample_size)
    bootstrap_delta.append(1 + 1 / np.mean([np.log(x) for x in bootstrap_sample]) - h_wave)

variation_row = sorted(bootstrap_delta)

t_1 = variation_row[int((1 - beta) * n_iterations / 2)]
t_2 = variation_row[int((1 + beta) * n_iterations / 2)]

lower_bound = h_wave - t_2
upper_bound = h_wave - t_1

print(f"Доверительный интервал: ({lower_bound}, {upper_bound})")
print(f"Длина: {upper_bound - lower_bound}")


Доверительный интервал: (1.8361556674371702, 2.272281073772345)
Длина: 0.43612540633517494


Параметрический бутстрап

In [5]:
n_iterations = 50000
beta = 0.95


h_wave = 1 + 1 / log_mean
bootstrap_delta = []

for _ in range(n_iterations):
    bootstrap_sample   =  []
    for _ in range(sample_size):
        y = rd.random()
        bootstrap_sample .append(F_inv(y, h_wave))

    bootstrap_delta.append(1 + 1 / np.mean([np.log(x) for x in bootstrap_sample ]) - h_wave)

variation_row = sorted(bootstrap_delta)

t_1 = variation_row[int((1 - beta) * n_iterations / 2)]
t_2 = variation_row[int((1 + beta) * n_iterations / 2)]


lower_bound = h_wave - t_2
upper_bound = h_wave - t_1

print(f"Доверительный интервал: ({lower_bound}, {upper_bound})")
print(f"Длина: {upper_bound - lower_bound}")


Доверительный интервал: (1.8445892992905244, 2.280542685724762)
Длина: 0.43595338643423753


#### f)  Сравнить все интервалы. 

Длина непараметрического бутстраповского доверительного интервала < Длина асимптотического доверительного интервала < Длина параметрического бутстраповского доверительного интервала